[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pyshka501/rl-textbook/blob/main/translations/ru/notebooks/ch01_introduction_ru.ipynb)

<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #e94560; font-size: 2.2em; margin: 0 0 10px 0;">Глава 1. Введение в обучение с подкреплением</h1>
  <p style="color: #a8dadc; font-size: 1.1em; margin: 0;">Взаимодействие агента и среды, стратегии, вознаграждения и компромисс между исследованием и эксплуатацией — на простом примере GridWorld.</p>
</div>

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #4fc3f7;">
  <strong style="color: #4fc3f7;">Подготовка окружения</strong><br>
  <span style="color: #cdd9e5;">Запустите ячейку ниже, чтобы установить зависимости. Работает в бесплатных версиях Google Colab и Kaggle — GPU не требуется.</span>
</div>

In [ ]:
!pip install gymnasium numpy matplotlib --quiet

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import random
from collections import defaultdict

# Reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'text.color': '#c9d1d9',
    'grid.color': '#21262d',
    'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d',
})
print('Setup complete.')

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #56d364;">
  <strong style="color: #56d364;">1. Базовые понятия обучения с подкреплением</strong>
</div>

Обучение с подкреплением (RL, reinforcement learning) — это вычислительная парадигма для обучения принятию решений через взаимодействие со средой. Ключевые компоненты:

| Компонент | Роль |
|-----------|------|
| **Агент** | Учится и принимает решения |
| **Среда** | Мир, с которым агент взаимодействует |
| **Состояние** $s_t$ | Текущая конфигурация среды |
| **Действие** $a_t$ | Выбор, сделанный агентом |
| **Вознаграждение** $r_t$ | Скалярный сигнал обратной связи |
| **Стратегия** $\pi(a|s)$ | Правило, ставящее в соответствие состоянию действие |

**Цикл взаимодействия агента и среды**: на каждом шаге $t$ агент наблюдает состояние $s_t$, выбирает действие $a_t \sim \pi(\cdot|s_t)$ и получает вознаграждение $r_{t+1}$ и следующее состояние $s_{t+1}$.

In [ ]:
# ─────────────────────────────────────────────
# GridWorld Environment (4x4)
# States: (row, col), Goal: (3,3), Hole: (1,1)
# Actions: 0=Up, 1=Down, 2=Left, 3=Right
# ─────────────────────────────────────────────

class GridWorld:
    """Simple 4x4 GridWorld with a goal and a hazard."""

    ACTIONS = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}  # Up/Down/Left/Right
    ACTION_NAMES = ['Up', 'Down', 'Left', 'Right']

    def __init__(self, size=4):
        self.size = size
        self.goal = (size - 1, size - 1)
        self.hole = (1, 1)
        self.state = None

    def reset(self):
        self.state = (0, 0)
        return self.state

    def step(self, action):
        dr, dc = self.ACTIONS[action]
        r, c = self.state
        nr = max(0, min(self.size - 1, r + dr))
        nc = max(0, min(self.size - 1, c + dc))
        self.state = (nr, nc)

        if self.state == self.goal:
            return self.state, +10.0, True   # Big reward at goal
        elif self.state == self.hole:
            return self.state, -5.0, True    # Penalty at hole
        else:
            return self.state, -0.1, False   # Small step cost

    def n_states(self):
        return self.size * self.size

    def state_index(self, s):
        return s[0] * self.size + s[1]


env = GridWorld(size=4)
state = env.reset()
print(f'Initial state: {state}')
next_state, reward, done = env.step(3)  # Move Right
print(f'After action Right → state={next_state}, reward={reward}, done={done}')

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #56d364;">
  <strong style="color: #56d364;">2. Визуализация GridWorld</strong>
</div>

Визуализируем сетку, чтобы понять структуру среды до запуска каких-либо агентов на ней.

In [ ]:
# ── Plot the GridWorld layout ──
fig, ax = plt.subplots(figsize=(5, 5))
ax.set_facecolor('#161b22')

for r in range(4):
    for c in range(4):
        color = '#21262d'
        text = f'({r},{c})'
        label = ''
        if (r, c) == (0, 0):
            color = '#1f6feb'
            label = 'START'
        elif (r, c) == (3, 3):
            color = '#2ea043'
            label = 'GOAL\n+10'
        elif (r, c) == (1, 1):
            color = '#da3633'
            label = 'HOLE\n-5'
        rect = mpatches.FancyBboxPatch(
            (c + 0.05, 3 - r + 0.05), 0.9, 0.9,
            boxstyle='round,pad=0.05', linewidth=1,
            edgecolor='#30363d', facecolor=color
        )
        ax.add_patch(rect)
        ax.text(c + 0.5, 3 - r + 0.5, label if label else text,
                ha='center', va='center', fontsize=10,
                color='white', fontweight='bold')

ax.set_xlim(0, 4)
ax.set_ylim(0, 4)
ax.set_xticks([])
ax.set_yticks([])
ax.set_title('4×4 GridWorld Environment', color='#c9d1d9', fontsize=14, pad=12)
plt.tight_layout()
plt.show()

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #56d364;">
  <strong style="color: #56d364;">3. Цикл взаимодействия агента и среды</strong>
</div>

Запустим две стратегии:
- **Случайная стратегия**: выбирает любое действие с равной вероятностью — $\pi(a|s) = 0{,}25$ для всех $a$.
- **Жадная стратегия**: всегда движется к цели (захардкоженная эвристика) — без исследования.

In [ ]:
# ── Policy definitions ──

def random_policy(state):
    """Uniform random policy — ignores state."""
    return np.random.randint(4)

def greedy_policy(state):
    """Simple heuristic: move right or down toward (3,3)."""
    r, c = state
    if c < 3 and r < 3:
        return np.random.choice([1, 3])  # Down or Right
    elif c < 3:
        return 3   # Right
    elif r < 3:
        return 1   # Down
    return np.random.randint(4)  # Already at goal


def run_episode(env, policy, max_steps=100):
    """Run one episode and return (states_visited, rewards)."""
    state = env.reset()
    states, rewards = [state], []
    for _ in range(max_steps):
        action = policy(state)
        next_state, reward, done = env.step(action)
        states.append(next_state)
        rewards.append(reward)
        state = next_state
        if done:
            break
    return states, rewards


# Single episode demo
np.random.seed(SEED)
states, rewards = run_episode(env, greedy_policy)
print('Greedy episode path:')
for i, (s, r) in enumerate(zip(states[:-1], rewards)):
    print(f'  Step {i+1}: state={s}  reward={r:.1f}')
print(f'  Final state: {states[-1]}  |  Total reward: {sum(rewards):.1f}')

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #56d364;">
  <strong style="color: #56d364;">4. Сравнение случайной и жадной стратегий</strong>
</div>

Запустим 200 эпизодов для каждой стратегии и сравним:
1. **Кумулятивное вознаграждение** за эпизод
2. **Длину эпизода** (число шагов)
3. **Долю эпизодов с достижением цели**

In [ ]:
# ── Run experiments: 200 episodes each ──
N_EPISODES = 200
np.random.seed(SEED)

results = {}
for name, policy in [('Random', random_policy), ('Greedy', greedy_policy)]:
    ep_rewards, ep_lengths, goal_reached = [], [], []
    for _ in range(N_EPISODES):
        states, rewards = run_episode(env, policy, max_steps=50)
        ep_rewards.append(sum(rewards))
        ep_lengths.append(len(rewards))
        goal_reached.append(states[-1] == env.goal)
    results[name] = {
        'rewards': ep_rewards,
        'lengths': ep_lengths,
        'goal_rate': np.mean(goal_reached),
    }

for name, data in results.items():
    print(f'{name:8s} | Avg reward: {np.mean(data["rewards"]):6.2f} | '
          f'Avg length: {np.mean(data["lengths"]):5.1f} | '
          f'Goal rate: {data["goal_rate"]*100:.1f}%')

In [ ]:
# ── Plot comparison ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = {'Random': '#e94560', 'Greedy': '#56d364'}
window = 20

# 1. Cumulative rewards (smoothed)
ax = axes[0]
for name, data in results.items():
    rewards = np.array(data['rewards'])
    smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
    ax.plot(smoothed, color=colors[name], label=name, linewidth=2)
ax.set_title('Episode Reward (20-ep moving avg)', fontsize=12)
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Episode length distribution
ax = axes[1]
for name, data in results.items():
    ax.hist(data['lengths'], bins=15, alpha=0.6, color=colors[name], label=name)
ax.set_title('Episode Length Distribution', fontsize=12)
ax.set_xlabel('Steps per Episode')
ax.set_ylabel('Count')
ax.legend()

# 3. Goal reach rate bar chart
ax = axes[2]
names = list(results.keys())
rates = [results[n]['goal_rate'] * 100 for n in names]
bars = ax.bar(names, rates, color=[colors[n] for n in names], width=0.4)
for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{rate:.1f}%', ha='center', va='bottom', fontsize=12, color='white')
ax.set_ylim(0, 110)
ax.set_title('Goal Reached Rate', fontsize=12)
ax.set_ylabel('% of Episodes')

plt.suptitle('Random Policy vs. Greedy Policy on GridWorld', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #56d364;">
  <strong style="color: #56d364;">5. Исследование и эксплуатация</strong>
</div>

Чистая жадность хрупка; чистая случайность неэффективна. **$\varepsilon$-жадная стратегия** балансирует обе крайности: с вероятностью $\varepsilon$ исследует случайно, иначе эксплуатирует лучшее известное действие. Посмотрим, как $\varepsilon$ влияет на качество.

In [ ]:
# ── Epsilon-greedy wrapper ──
def make_epsilon_greedy(greedy_fn, epsilon):
    def policy(state):
        if np.random.rand() < epsilon:
            return np.random.randint(4)   # Explore
        return greedy_fn(state)           # Exploit
    return policy


np.random.seed(SEED)
epsilons = [0.0, 0.1, 0.3, 0.5, 1.0]
eps_results = {}

for eps in epsilons:
    policy = make_epsilon_greedy(greedy_policy, eps)
    ep_rewards = []
    for _ in range(N_EPISODES):
        _, rewards = run_episode(env, policy, max_steps=50)
        ep_rewards.append(sum(rewards))
    eps_results[eps] = ep_rewards


# ── Plot: avg reward vs epsilon ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
cmap = plt.cm.plasma
for i, eps in enumerate(epsilons):
    color = cmap(i / len(epsilons))
    smoothed = np.convolve(eps_results[eps], np.ones(window)/window, mode='valid')
    ax.plot(smoothed, color=color, label=f'ε={eps}', linewidth=2)
ax.set_title('Reward per Episode by ε', fontsize=12)
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward (smoothed)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[1]
avg_rewards = [np.mean(eps_results[eps]) for eps in epsilons]
ax.plot(epsilons, avg_rewards, 'o-', color='#58a6ff', linewidth=2, markersize=8)
for eps, r in zip(epsilons, avg_rewards):
    ax.annotate(f'{r:.2f}', (eps, r), textcoords='offset points',
                xytext=(0, 10), ha='center', color='#c9d1d9', fontsize=9)
ax.set_title('Mean Episode Reward vs. ε', fontsize=12)
ax.set_xlabel('Epsilon (exploration rate)')
ax.set_ylabel('Mean Total Reward')
ax.grid(True, alpha=0.3)

plt.suptitle('Exploration-Exploitation Trade-off', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

<div style="background: #0d2137; padding: 20px; border-radius: 8px; border: 1px solid #1f6feb; margin-top: 20px;">
  <h3 style="color: #79c0ff; margin-top: 0;">Глава 1 — ключевые выводы</h3>
  <ul style="color: #c9d1d9; line-height: 1.8;">
    <li>RL формализуется как <strong>цикл взаимодействия агента и среды</strong>: наблюдать состояние, выбрать действие, получить вознаграждение.</li>
    <li><strong>Стратегия</strong> $\pi(a|s)$ — это правило принятия решений агентом.</li>
    <li><strong>Вознаграждения</strong> — скалярные сигналы; цель — максимизировать кумулятивное вознаграждение (отдачу).</li>
    <li><strong>Случайная стратегия</strong> редко находит цель; <strong>жадная</strong> уверенно достигает цели, но не исследует.</li>
    <li><strong>$\varepsilon$-жадная стратегия</strong> интерполирует: малое $\varepsilon$ ≈ эксплуатация, большое $\varepsilon$ ≈ исследование. Оптимальное $\varepsilon$ зависит от задачи.</li>
  </ul>
</div>